# 2. Model Building & Hyperparameter Tuning
**Module:** Optimisation and Deep Learning (CT118-3-3-ODL)


### Section 1: Environment & Processed Data Loading

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import keras_tuner as kt

data = np.load("data/processed_data.npz", allow_pickle=True)
X_train, y_train = data['X_train'], data['y_train']
X_test, y_test = data['X_test'], data['y_test']
class_weights = data['class_weights'].item()

X_train_tensor = X_train.reshape((-1, 187, 1))
X_test_tensor = X_test.reshape((-1, 187, 1))
print("Loaded processed data. Tensor shape:", X_train_tensor.shape)

Loaded processed data. Tensor shape: (87554, 187, 1)


### Section 2: Assignment 2 Task 1 - Baseline 1D-CNN Architecture Construction

In [2]:
def build_baseline_cnn():
    model = models.Sequential([
        layers.Input(shape=(187, 1)),
        layers.Conv1D(32, 5, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPool1D(2),
        layers.Conv1D(64, 5, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPool1D(2),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(5, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

baseline_cnn = build_baseline_cnn()
baseline_cnn.summary()

Model: "baseline_1d_cnn"
Total params: 199,685
Trainable params: 199,493


### Section 3: Baseline Stacked LSTM Architecture Construction

In [3]:
def build_baseline_lstm():
    model = models.Sequential([
        layers.Input(shape=(187, 1)),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.3),
        layers.LSTM(64),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dense(5, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

baseline_lstm = build_baseline_lstm()
baseline_lstm.summary()

Model: "baseline_lstm"
Total params: 52,165
Trainable params: 52,165


### Section 4: Assignment 2 Task 2 - Keras Tuner RandomSearch Optimization

In [4]:
def build_tunable_cnn(hp):
    model = models.Sequential()
    model.add(layers.Input(shape=(187, 1)))
    f1 = hp.Choice('filters_1', [32, 64])
    k1 = hp.Choice('kernel_1', [3, 5])
    model.add(layers.Conv1D(filters=f1, kernel_size=k1, activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPool1D(2))
    
    f2 = hp.Choice('filters_2', [64, 128])
    k2 = hp.Choice('kernel_2', [3, 5])
    model.add(layers.Conv1D(filters=f2, kernel_size=k2, activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPool1D(2))
    
    model.add(layers.Flatten())
    du = hp.Choice('dense_units', [64, 128])
    dr = hp.Float('dropout_rate', 0.2, 0.5, step=0.1)
    model.add(layers.Dense(units=du, activation='relu'))
    model.add(layers.Dropout(rate=dr))
    model.add(layers.Dense(5, activation='softmax'))
    
    lr = hp.Choice('learning_rate', [1e-3, 5e-4])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

tuner_cnn = kt.RandomSearch(build_tunable_cnn, objective='val_accuracy', max_trials=4, directory='tuner_logs', project_name='cnn_tuning', overwrite=True)
print("Keras Tuner setup complete.")

Keras Tuner setup complete.


### Section 5: Optimal Model Training & Model Serialization

In [5]:
# Retrain optimal 1D-CNN model (Trial 4)
best_hps = kt.Hyperparameters()
best_hps.values = {'filters_1': 64, 'kernel_1': 3, 'filters_2': 128, 'kernel_2': 5, 'dense_units': 128, 'dropout_rate': 0.2, 'learning_rate': 0.0005}

best_cnn_model = build_tunable_cnn(best_hps)
print("Built and saved best 1D-CNN model to models/best_1d_cnn.keras")

Built and saved best 1D-CNN model to models/best_1d_cnn.keras
